In [6]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from fnmatch import fnmatch

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error
from keras import models, layers, optimizers, 
from keras.callbacks import EarlyStopping, ReduceLROnPlateau
# from tf.keras.models import Sequential
# from tensorflow.keras.layers import Dense, Dropout
# from tensorflow.keras.optimizers import Adam
# from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
import joblib
import os

In [7]:
from pathlib import Path
from fnmatch import fnmatch
import pandas as pd

root = (Path.cwd() / "apt_prices_poland")
if not root.exists():
    root = Path.cwd().parent / "apt_prices_poland"  

skip_pattern = "apartments_rent_pl_*.csv"

csv_paths = [
    path
    for path in root.rglob("*.csv")
    if not fnmatch(path.name, skip_pattern)
]

if not csv_paths:
    raise FileNotFoundError(f"No CSV files found under {root}")

frames = [pd.read_csv(p) for p in csv_paths]
combined = pd.concat(frames, ignore_index=True)


In [12]:
df = combined.copy()

In [13]:
df["price_per_sqm"] = df["price"] / np.clip(df["squareMeters"], 1e-6, None)

# sanity/outlier filters (tweak if needed)
# df = df[(df["squareMeters"] >= 10) & (df["squareMeters"] <= 4000)]
# df = df[(df["price_per_sqm"] > 500) & (df["price_per_sqm"] < 100_000)]


In [14]:
num_feats = [
    "squareMeters","rooms","floor","floorCount","buildYear",
    "latitude","longitude","centreDistance","poiCount",
    "schoolDistance","clinicDistance","postOfficeDistance",
    "kindergartenDistance","restaurantDistance","collegeDistance","pharmacyDistance"
]
cat_low = ["type","ownership"]  # low-cardinality categoricals
id_col = "id"
target_col = "price_per_sqm"

In [15]:

available = [c for c in num_feats + cat_low + [id_col, target_col] if c in df.columns]
df = df[available].copy()


In [16]:
for c in cat_low:
    if c in df.columns:
        df[c] = df[c].astype("string").fillna("Unknown")

In [ ]:
groups = df[id_col].values

gss1 = GroupShuffleSplit(n_splits=1, test_size=0.15, random_state=42)
train_idx, test_idx = next(gss1.split(df, groups=groups))
df_train = df.iloc[train_idx].reset_index(drop=True)
df_test  = df.iloc[test_idx].reset_index(drop=True)


In [19]:
# From the remaining training set, carve out a validation set (again group-wise)
groups_tr = df_train[id_col].values
gss2 = GroupShuffleSplit(n_splits=1, test_size=0.1765, random_state=42)  # ~0.15/0.85 → final split 70/15/15
tr_idx, val_idx = next(gss2.split(df_train, groups=groups_tr))
df_tr  = df_train.iloc[tr_idx].reset_index(drop=True)
df_val = df_train.iloc[val_idx].reset_index(drop=True)

X_tr  = df_tr[num_feats + cat_low]
y_tr  = df_tr[target_col].values
X_val = df_val[num_feats + cat_low]
y_val = df_val[target_col].values
X_te  = df_test[num_feats + cat_low]
y_te  = df_test[target_col].values

In [20]:
#Preprocessing (impute + scale numerics; impute + one-hot categoricals)
numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

pre = ColumnTransformer(
    transformers=[
        ("num", numeric_pipeline, [c for c in num_feats if c in X_tr.columns]),
        ("cat", categorical_pipeline, [c for c in cat_low if c in X_tr.columns]),
    ],
    remainder="drop"
)

In [21]:
# Fit on train, transform all
X_tr_p  = pre.fit_transform(X_tr)
X_val_p = pre.transform(X_val)
X_te_p  = pre.transform(X_te)

print("Train shape:", X_tr_p.shape, "Val shape:", X_val_p.shape, "Test shape:", X_te_p.shape)


Train shape: (136605, 23) Val shape: (29775, 23) Test shape: (29188, 23)


In [ ]:
# === 4) Build MLP model
def build_mlp(input_dim: int) -> models.Sequential:
    model = models.Sequential([
        layers.Dense(256, activation="relu", input_shape=(input_dim,)),
        layers.Dropout(0.25),
        layers.Dense(128, activation="relu"),
        layers.Dropout(0.15),
        layers.Dense(64, activation="relu"),
        layers.Dense(1)  # regression
    ])
    model.compile(optimizer=optimizers.Adam(learning_rate=1e-3), loss="mse", metrics=["mae"])
    return model

model = build_mlp(X_tr_p.shape[1])

callbacks = [
    EarlyStopping(monitor="val_mae", patience=10, restore_best_weights=True),
    ReduceLROnPlateau(monitor="val_mae", factor=0.5, patience=5, min_lr=1e-5)
]

history = model.fit(
    X_tr_p, y_tr,
    validation_data=(X_val_p, y_val),
    epochs=100,
    batch_size=1024,
    callbacks=callbacks,
    verbose=1
)


/Users/heesung/Documents/Poland/AGH/2025_Winter/Diploma/Raw/.venv/lib/python3.12/site-packages/keras/src/layers/core/dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


AttributeError: 'list' object has no attribute 'EarlyStopping'